In [0]:
def fiche(case_id):
    base = "workspace.forecast_debug"
    gap = spark.table(f"{base}.cmp_forecast_gap").filter(f"case_id = '{case_id}'").first()
    fr = spark.table(f"{base}.cmp_freshness").filter(f"case_id = '{case_id}'").first()
    diffs = spark.table(f"{base}.cmp_inputs").filter(
        f"case_id = '{case_id}' AND is_difference = true").collect()

    print(f"===== {case_id} =====")
    print(f"1. Écart      : {gap['batch_forecast']:.0f} (Batch) vs {gap['realtime_forecast']:.0f} (Real-Time) "
          f"= {gap['difference']:.0f} unités ({gap['difference_pct']} %) "
          f"| dans la tolérance de 5 % ? {gap['within_tolerance']}")
    print(f"2. Fraîcheur  : âge Batch {fr['batch_age_hours']:.1f} h, "
          f"âge Real-Time {fr['realtime_age_hours']:.1f} h -> {fr['freshness_status']} (seuil obsolète : 24 h)")
    print(f"5. Warning    : {fr['batch_warning'] or 'aucun'}")
    feats = [d for d in diffs if d["finding_type"] == "FEATURE"]
    cfgs = [d for d in diffs if d["finding_type"] == "CONFIG"]
    print("3. Features qui diffèrent :",
          [f"{d['name']}: {d['batch_value']} vs {d['realtime_value']}" for d in feats] or "aucune")
    print("4. Configs qui diffèrent  :",
          [f"{d['name']}: {d['batch_value']} vs {d['realtime_value']}" for d in cfgs] or "aucune")
    print()

for c in ["CASE_003", "CASE_007", "CASE_009"]:
    fiche(c)